In [1]:
import os
import json
import torch
import re
from transformers import pipeline
from tqdm import tqdm

KeyboardInterrupt: 

In [ ]:
device = 0 if torch.cuda.is_available() else -1

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=device,
    chunk_length_s=25,
    stride_length_s=5
)

print("ASR Ready")

In [ ]:
def clean_text(text):
    text = re.sub(r"[^\x00-\x7F]+", "", text)  # remove non-English
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'(\b\w+\b)( \1\b)+', r'\1', text)
    return text.strip()

In [ ]:
audio_path = "ES2003b.Mix-Headset.wav"

result = asr(
    audio_path,
    return_timestamps=True,
    generate_kwargs={"language": "en"}
)

segments = []

for seg in result["chunks"]:
    start, end = seg["timestamp"]
    
    if start is None or end is None:
        continue

    segments.append(clean_text(seg["text"]))

full_text = " ".join(segments)

print("Transcription Done")

In [ ]:
print(full_text[:1500])

In [ ]:
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=device
)

print("Summarizer Ready")

In [ ]:
def chunk_text(text, max_words=400):
    words = text.split()
    chunks = []

    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i+max_words])
        chunks.append(chunk)

    return chunks

chunks = chunk_text(full_text)

print(f"Total chunks: {len(chunks)}")

In [ ]:
summaries = []

for chunk in chunks:
    summary = summarizer(chunk, max_length=120, min_length=40, do_sample=False)[0]["summary_text"]
    summaries.append(summary)

In [ ]:
final_summary = " ".join(summaries)
print(final_summary)

In [ ]:
with open("final_summary.txt", "w", encoding="utf-8") as f:
    f.write(final_summary)

print("Saved as final_summary.txt")

In [ ]:
!pip install pyannote.audio==2.1.1

In [ ]:
import os
from pyannote.audio import Pipeline

HUGGINGFACE_TOKEN = os.environ["HF_TOKEN"]  # set via .env / environment — never hardcode tokens

diar_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization",
    use_auth_token=HUGGINGFACE_TOKEN
)

print("Diarization model loaded")

In [2]:
pip list | findstr numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install numpy==1.26.4

  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl (15.8 MB)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.37.0 which is incompatible.
